In [2]:
import torch
from torch import nn
from torch.nn import functional as F

In [3]:
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))

In [4]:
X = torch.rand(2, 20)
print(X)

tensor([[0.6708, 0.1318, 0.7330, 0.4978, 0.7984, 0.0220, 0.1713, 0.2234, 0.6373,
         0.4554, 0.9875, 0.8365, 0.5805, 0.8997, 0.9142, 0.9097, 0.1822, 0.2600,
         0.5949, 0.6138],
        [0.6740, 0.7741, 0.9653, 0.2499, 0.7647, 0.3000, 0.2347, 0.2451, 0.7088,
         0.2382, 0.6224, 0.2956, 0.9687, 0.1293, 0.1710, 0.2705, 0.0363, 0.7660,
         0.6928, 0.6439]])


In [5]:
Y = net(X)
print(Y)
print(Y.shape)

tensor([[-0.0629,  0.0051,  0.2552, -0.2720, -0.1774,  0.0784, -0.2800,  0.1585,
         -0.0472,  0.0505],
        [ 0.0190,  0.0115,  0.0960, -0.2588, -0.1795,  0.0807, -0.2780,  0.1908,
         -0.1285,  0.1547]], grad_fn=<AddmmBackward0>)
torch.Size([2, 10])


In [6]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.LazyLinear(256)
        self.out = nn.LazyLinear(10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))


In [7]:
net = MLP()
Y = net(X)
print(Y)
print(Y.shape)

tensor([[ 0.0389,  0.0829,  0.0942,  0.2001,  0.2180,  0.1289, -0.0953,  0.0533,
          0.0731, -0.0843],
        [ 0.0088,  0.1333,  0.1183,  0.1765,  0.1580, -0.0005, -0.0401,  0.1925,
          0.1212, -0.1775]], grad_fn=<AddmmBackward0>)
torch.Size([2, 10])


In [8]:
class MySequienal(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, value in enumerate(args):
            self.add_module(str(idx), value)

    def forward(self, input_x):
        for module in self.children():
            input_x = module(input_x)
        return input_x


In [9]:
net = MySequienal(nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10)))
Y = net(X)
print(Y)
print(Y.shape)

tensor([[ 0.0275, -0.5838, -0.0129,  0.2532,  0.0281, -0.2387,  0.1752, -0.0518,
          0.3001, -0.0856],
        [-0.0131, -0.4474,  0.0040,  0.1402,  0.0292, -0.1901,  0.1525, -0.0736,
          0.2385, -0.1265]], grad_fn=<AddmmBackward0>)
torch.Size([2, 10])


In [10]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.c = torch.rand(20, 20)
        self.linear = nn.LazyLinear(20)

    def forward(self, input_x):
        input_x = self.linear(input_x)
        input_x = F.relu(input_x @ self.c + 1)
        input_x = self.linear(input_x)
        return input_x


In [11]:
net = FixedHiddenMLP()
Y = net(X)
print(Y)

tensor([[ 0.6023,  0.1981,  0.0745, -0.1784,  0.1946,  0.5410,  0.3711,  1.4652,
         -0.4813,  1.1474,  0.8065, -0.7962, -0.9384, -0.7503, -0.2588, -0.4038,
          0.3078,  0.2683,  0.1393, -0.4167],
        [ 0.5895,  0.0191,  0.1504, -0.4579,  0.2635,  0.2616,  0.2789,  1.2334,
         -0.3924,  1.1267,  0.7404, -0.7534, -0.8191, -0.7788, -0.0507, -0.3266,
          0.3219,  0.1444,  0.1639, -0.3671]], grad_fn=<AddmmBackward0>)


In [12]:
################### paramters management########################

In [13]:
net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(), nn.LazyLinear(1))
print(net)

Sequential(
  (0): LazyLinear(in_features=0, out_features=8, bias=True)
  (1): ReLU()
  (2): LazyLinear(in_features=0, out_features=1, bias=True)
)


In [14]:
X = torch.rand(2, 3)
print(X)
Y = net(X)
print(Y)

tensor([[0.3020, 0.0428, 0.4899],
        [0.6521, 0.5106, 0.7764]])
tensor([[-0.1546],
        [-0.2635]], grad_fn=<AddmmBackward0>)


In [15]:
print(net[1].state_dict())
print(net[0].state_dict())
print(net[2].state_dict())

OrderedDict()
OrderedDict([('weight', tensor([[ 0.5163, -0.2244,  0.3020],
        [ 0.2321, -0.2889,  0.2599],
        [ 0.3680,  0.2806,  0.2242],
        [ 0.2848,  0.0585, -0.5761],
        [-0.2569,  0.1693,  0.2402],
        [ 0.5263, -0.1091,  0.1446],
        [-0.0733,  0.4130,  0.4061],
        [-0.1486, -0.4374,  0.4699]])), ('bias', tensor([-0.1517, -0.4684, -0.3479,  0.0739, -0.3513,  0.5316,  0.5070,  0.5282]))])
OrderedDict([('weight', tensor([[-0.2854,  0.3241, -0.2511, -0.1977,  0.2264, -0.0880,  0.1397,  0.2806]])), ('bias', tensor([-0.3403]))])


In [16]:
for name, params in net.named_parameters():
    print("name:" + str(name))
    print("params:" + str(params))

name:0.weight
params:Parameter containing:
tensor([[ 0.5163, -0.2244,  0.3020],
        [ 0.2321, -0.2889,  0.2599],
        [ 0.3680,  0.2806,  0.2242],
        [ 0.2848,  0.0585, -0.5761],
        [-0.2569,  0.1693,  0.2402],
        [ 0.5263, -0.1091,  0.1446],
        [-0.0733,  0.4130,  0.4061],
        [-0.1486, -0.4374,  0.4699]], requires_grad=True)
name:0.bias
params:Parameter containing:
tensor([-0.1517, -0.4684, -0.3479,  0.0739, -0.3513,  0.5316,  0.5070,  0.5282],
       requires_grad=True)
name:2.weight
params:Parameter containing:
tensor([[-0.2854,  0.3241, -0.2511, -0.1977,  0.2264, -0.0880,  0.1397,  0.2806]],
       requires_grad=True)
name:2.bias
params:Parameter containing:
tensor([-0.3403], requires_grad=True)


In [17]:
######################paramters init#################

In [18]:
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))
X = torch.rand((4, 5))
print(X)
Y = net(X)
print(Y)
print(Y.shape)

tensor([[0.3641, 0.1378, 0.3780, 0.6018, 0.6434],
        [0.1636, 0.7339, 0.6703, 0.0282, 0.0949],
        [0.4064, 0.8373, 0.8341, 0.2189, 0.2517],
        [0.5905, 0.1241, 0.6502, 0.3209, 0.1717]])
tensor([[-0.0979, -0.1938, -0.0488, -0.0347, -0.1066, -0.1487, -0.3578, -0.0126,
          0.1687,  0.0297],
        [ 0.0884, -0.1985,  0.1125, -0.0820, -0.1852, -0.0499, -0.4276, -0.0303,
          0.0974,  0.0700],
        [ 0.1461, -0.2667,  0.0845, -0.0323, -0.1670, -0.0698, -0.4449, -0.0761,
          0.1581,  0.1083],
        [ 0.0139, -0.1834,  0.0388, -0.0414, -0.0886, -0.0867, -0.4125, -0.0412,
          0.0768,  0.0577]], grad_fn=<AddmmBackward0>)
torch.Size([4, 10])


In [19]:
def init_model(module):
    print("init_model been called:" + str(module))
    if type(module) == nn.LazyLinear:
        nn.init.normal_(module.weight, mean=0, std=0.01)
        nn.init.zeros_(module.bias)



In [20]:
net.apply(init_model)

init_model been called:Linear(in_features=5, out_features=256, bias=True)
init_model been called:ReLU()
init_model been called:Linear(in_features=256, out_features=10, bias=True)
init_model been called:Sequential(
  (0): Linear(in_features=5, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=10, bias=True)
)


Sequential(
  (0): Linear(in_features=5, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=10, bias=True)
)

In [22]:
print(net[0].weight.data)

tensor([[-0.2521, -0.4411, -0.1336, -0.4336,  0.2722],
        [-0.2399, -0.4133,  0.3841, -0.3487, -0.1123],
        [-0.3876, -0.3375, -0.3360,  0.2213, -0.1900],
        ...,
        [ 0.4049, -0.3665,  0.1351, -0.4020, -0.2499],
        [-0.3651,  0.2229, -0.0264, -0.1406, -0.2076],
        [-0.0563,  0.2756,  0.3772, -0.3185,  0.0432]])


In [23]:
def init_kaiming(module):
    nn.init.kaiming_normal_(module.weight)
    nn.init.zeros_(module.bias)

In [24]:
net.apply(init_model)
print(net[0].weight.data)

init_model been called:Linear(in_features=5, out_features=256, bias=True)
init_model been called:ReLU()
init_model been called:Linear(in_features=256, out_features=10, bias=True)
init_model been called:Sequential(
  (0): Linear(in_features=5, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=10, bias=True)
)
tensor([[-0.2521, -0.4411, -0.1336, -0.4336,  0.2722],
        [-0.2399, -0.4133,  0.3841, -0.3487, -0.1123],
        [-0.3876, -0.3375, -0.3360,  0.2213, -0.1900],
        ...,
        [ 0.4049, -0.3665,  0.1351, -0.4020, -0.2499],
        [-0.3651,  0.2229, -0.0264, -0.1406, -0.2076],
        [-0.0563,  0.2756,  0.3772, -0.3185,  0.0432]])


In [ ]:
class MyLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.parameter(torch.rand())

In [25]:
torch.save(X, "X_input.pth")
torch.save(Y, "Y_input.pth")